# 040. Praca z DataFrame: Selekcja, Czyszczenie i Indeksowanie

Kluczowy moduł operacji tabelarycznych: diagnostyka (`.info()`, `.describe()`, `.shape`, `.dtypes`, `.nunique()`), zarządzanie indeksem, czyszczenie braków z parametrem **`subset`**, imputacja `.fillna()`, usuwanie duplikatów, sortowanie wielokolumnowe oraz idiomatyczna iteracja z `itertuples()`.


## 1. Tworzenie tabeli i pełna diagnostyka


In [ ]:
import pandas as pd
import numpy as np


In [ ]:

df = pd.DataFrame({
    'Klient': ['Jan Kowalski', 'Anna Nowak', 'Piotr Wiśniewski', 'Maria Dąbrowska', 'Tomasz Lis'],
    'Wiek': [28, 42, 35, 22, 51],
    'Miasto': ['Warszawa', 'Kraków', 'Warszawa', 'Gdańsk', 'Kraków'],
    'Kwota_Zakupow': [1250.50, 4300.00, 890.20, 310.00, 2150.00],
    'Status': ['Aktywny', 'VIP', 'Aktywny', 'Nowy', 'VIP']
})

print("Wymiary (.shape):", df.shape)
print("Kolumny (.columns):", df.columns.tolist())
print("\nTypy danych (.dtypes):\n", df.dtypes)
print("\nLiczba unikalnych wartości per kolumna (.nunique()):\n", df.nunique())
print("\nPodsumowanie numeryczne (.describe()):\n", df.describe())
print("\nPodsumowanie wszystkich kolumn (.describe(include='all')):\n", df.describe(include='all'))


## 2. Czyszczenie braków danych: `dropna()` i kluczowy parametr `subset`


In [ ]:
df_zamowienia = pd.DataFrame({
    'zamowienie_id': [101, 102, 103, 104, 105],
    'klient': ['Jan', 'Anna', 'Piotr', None, 'Ewa'],
    'kwota': [1250.0, 430.0, np.nan, 890.0, np.nan],
    'telefon': ['601-200-300', None, None, '500-100-200', None]
})


In [ ]:
print("Wykrywanie braków per kolumna (.isnull().sum()):")
print(df_zamowienia.isnull().sum())

# PUŁAPKA: dropna() bez parametrów usuwa prawie wszystko!
print(f"\nDomyślne dropna() zostawiło {len(df_zamowienia.dropna())} wierszy.")


In [ ]:

# PRAWIDŁOWO: Usuwamy tylko gdy brak w kluczowej kolumnie (kwota)
czyste_kwoty = df_zamowienia.dropna(subset=['kwota'], ignore_index=True)
print("\nPo usunięciu braków tylko w kwocie (subset=['kwota']):")
print(czyste_kwoty)


In [ ]:

# Wypełnianie braków słownikiem per kolumna (.fillna):
df_wypelnione = df_zamowienia.fillna({
    'klient': 'Anonim',
    'kwota': df_zamowienia['kwota'].median(),
    'telefon': 'Brak telefonu'
})
print("\nPo wypełnieniu słownikiem (.fillna):")
print(df_wypelnione)


## 3. Usuwanie i audyt duplikatów: `drop_duplicates` z parametrem `subset`


In [ ]:
df_logi = pd.DataFrame({
    'user_id': [1, 2, 2, 3, 4, 4],
    'nazwisko': ['Kowalski', 'Nowak', 'Nowak-Aktualizacja', 'Wiśniewski', 'Dąbrowska', 'Dąbrowska'],
    'status': ['Aktywny', 'Logowanie', 'Wylogowanie', 'Aktywny', 'Aktywny', 'Aktywny']
})


In [ ]:

# KROK 1: Audyt powtórzeń przed usunięciem (keep=False pokazuje wszystkie duplikaty!)
print("Wykryte powtórzenia użytkowników:")
print(df_logi[df_logi.duplicated(subset=['user_id'], keep=False)])


In [ ]:

# KROK 2: Usunięcie duplikatów z zachowaniem najświeższego wpisu (keep='last')
df_dedup = df_logi.drop_duplicates(subset=['user_id'], keep='last', ignore_index=True)
print("\nPo deduplikacji z keep='last':")
print(df_dedup)


## 4. Sortowanie wielokolumnowe (`sort_values`)


In [ ]:
# Sortowanie: Miasto rosnąco (A-Z), a w ramach miasta Kwota malejąco (od największej)
df_posortowane = df.sort_values(
    by=['Miasto', 'Kwota_Zakupow'],
    ascending=[True, False],
    ignore_index=True
)
print("Sortowanie wielokolumnowe (Miasto A-Z, Kwota Z-A):")
print(df_posortowane[['Miasto', 'Klient', 'Kwota_Zakupow']])


## 5. Zliczanie wartości i badanie rozkładów (`value_counts`)


In [ ]:
print("Rozkład miast (liczba klientów):")
print(df['Miasto'].value_counts())


In [ ]:

print("\nUdziały procentowe statusów (normalize=True):")
print((df['Status'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')


In [ ]:

print("\nWspółwystępowanie par: Miasto vs Status:")
print(df.value_counts(subset=['Miasto', 'Status']))


## 6. Bezpieczna i wydajna iteracja po wierszach: `itertuples()`


In [ ]:
# ANTYWZORZEC: for idx, row in df.iterrows() -> powolne i gubi typy int!
# PRAWIDŁOWO: for row in df.itertuples() -> szybkie namedtuple, zachowuje dtypes:
print("Iteracja po wierszach za pomocą itertuples():")
for row in df.itertuples(index=True):
    print(f"[{row.Index}] Klient: {row.Klient} | Wiek: {row.Wiek} (typ: {type(row.Wiek).__name__}) | Kwota: {row.Kwota_Zakupow:.2f} zł")
